## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
import os

# from langchain_ollama import ChatOllama
# from langchain_anthropic import ChatAnthropic

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [2]:
MODEL = "openrouter/free"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [4]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(
    model=MODEL,
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    temperature=0
)

### These LangChain objects implement the method `invoke()`

In [5]:
retriever.invoke("Who is Avery?")

[Document(id='e871860c-2ff7-4b01-b30e-ec0239aa7398', metadata={'source': 'knowledge-base/employees/Avery Lancaster.md', 'doc_type': 'employees'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and ada

In [6]:
llm.invoke("Who is Avery?")

AIMessage(content='\n\nAvery is a common English name that can be a given name or a surname. Without more context, it could refer to many different people. Here are a few possibilities:\n\n- **Avery** as a first name: It\'s been used for both boys and girls, though it\'s more common for girls in recent years.\n- **Notable people**: There\'s Avery, the American singer from the duo *Avery*; **Avery Jenkins**, a professional disc golfer; **Avery Schreiber**, a comedian; and **Avery Brundage**, the former president of the International Olympic Committee.\n- **Fictional characters**: For example, **Avery** from the TV show *The Fosters*, or **Avery** in the video game *Life is Strange*.\n- **As a surname**: The name has English origins, possibly meaning "elf counselor" or "ruler of the elves."\n\nIf you have a specific Avery in mind—like a celebrity, a character, or someone from a particular field—let me know, and I can give you more details!', additional_kwargs={'refusal': None}, response_

## Time to put this together!

In [7]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [8]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [9]:
answer_question("Who is Averi Lancaster?", [])

'**Avery Lancaster** is the Co‑Founder and Chief Executive Officer (CEO) of Insurellm. Here’s a quick snapshot of who she is:\n\n- **Date of Birth:** March\u202f15,\u202f1985  \n- **Current Role:** Co‑Founder & CEO (since the company’s inception in 2015)  \n- **Location:** San\u202fFrancisco, California  \n- **Current Salary:** $225,000  \n\n**Career Highlights**\n- **2015\u202f–\u202fPresent:** Co‑founded Insurellm and has guided the company to become a leading Insurance‑Tech provider, leveraging innovative leadership and risk‑management expertise.  \n- **2013\u202f–\u202f2015:** Served as a **Senior Product Manager** at Innovate Insurance Solutions, where she developed groundbreaking insurance products for the tech sector.  \n\nAvery is known for her strategic vision, risk‑management acumen, and ability to drive Insurellm into the mainstream insurance market. If you need more details about her background, leadership approach, or any other aspect of her role, just let me know!'

## What could possibly come next? 😂

In [10]:
gr.ChatInterface(answer_question).launch()

/Users/mukbul/Work/AI/Projects/courses/ed-donner/llm_engineering/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Admit it - you thought RAG would be more complicated than that!!